In [2]:
import pyvisa as pv
import time
import matplotlib.pyplot as plt
import json 
import requests
import pandas as pd
import numpy as np
import os
import qcodes as qc

%matplotlib tk

In [3]:
from qcodes.dataset import (Measurement, load_or_create_experiment, plot_by_id, initialise_or_create_database_at, experiments, load_by_run_spec)


In [4]:
path2database = "mx_cal.db"
initialise_or_create_database_at(path2database)
load_or_create_experiment(experiment_name="mx_cal", sample_name="out")

mx_cal#out#1@c:\Users\franco\Documents\Resonators\mx_cal.db
-----------------------------------------------------------
1-results-1-None-0
2-results-2-None-0
3-results-3-None-0
4-results-4-LO_amp,LO_freq,I_freq,Q_freq,I_amp,Q_amp-0
5-results-5-sa_sa_freq_axis,sa_sa_trace,LO_amp,LO_freq,I_freq,Q_freq,I_amp,Q_amp-7
6-results-6-sa_sa_freq_axis,sa_sa_trace,LO_amp,LO_freq,I_freq,Q_freq,I_amp,Q_amp-7

#### generador IF

In [5]:
rm=pv.ResourceManager()
gen=rm.open_resource(f"TCPIP::192.168.1.94::inst0::INSTR")


### generador RF

In [6]:
from qcodes.instrument_drivers.Keysight import KeysightN5173B
vsg = KeysightN5173B('cgx', 'TCPIP0::192.168.1.23::inst0::INSTR')

Connected to: Keysight Technologies N5166B (serial:MY61253319, firmware:B.01.96) in 0.02s


### spectrum analyzer

In [7]:
from qcodes.instrument_drivers.Keysight import KeysightN9030B
sa = KeysightN9030B("sa", "GPIB0::18::INSTR")
sa.cont_meas(True)  

Connected to: Keysight Technologies N9032B (serial:MY62500121, firmware:A.34.06) in 0.28s


In [8]:
station = qc.Station(sa)
print(station.components)

{'sa': <KeysightN9030B: sa>}


### amplitude and phase settings

In [9]:
### setting amplitud and phase
channel1 = 1
channel2 = 2
amplitude1 = 0.1
amplitude2 = 0.1
pha1 = 180
pha2 = 90

freq1 = 200e3
freq2 = 200e3
gen.write(f"SOUR{channel1}:VOLT {amplitude1}")
gen.write(f"SOUR{channel2}:VOLT {amplitude2}")

gen.write(f"SOUR{channel1}:PHAS {pha1}")
gen.write(f"SOUR{channel2}:PHAS {pha2}")

gen.write(f"SOUR{channel1}:FREQ {freq1}")
gen.write(f"SOUR{channel2}:FREQ {freq2}")

21

## frequency sweep

sweep IF

In [10]:
f_lo = 4e9#5.096600e9
vsg.frequency(f_lo)

In [68]:
fstart, fstop, step = 200e3,300e3, 1e3
freqs = np.arange(fstart, fstop+step, step)
print(len(freqs))

for f in freqs:
    gen.write(f"SOUR{channel1}:FREQ {f}")
    gen.write(f"SOUR{channel2}:FREQ {f}")
    
    time.sleep(1)

101


sweep LO

In [75]:
### fixing IF freqs
f_if = 200e3
gen.write(f"SOUR{channel1}:FREQ {f_if}")
gen.write(f"SOUR{channel2}:FREQ {f_if}")

21

In [61]:
power_vsg = 16  # Power in dBm
vsg.power(power_vsg)


In [11]:
f_if = 100e3

In [39]:
f_res = 5.09620e9
vsg.frequency(f_res)
f_lo = f_res - 2*f_if

In [ ]:
fstart_lo, fstop_lo, step_lo = 5.096600e9-2*f_if, 5.096700e9-2*f_if, 2e3
fstart_lo, fstop_lo, step_lo = 5.0960e9, 5.096400e9, 5e3
freqs_lo = np.arange(fstart_lo, fstop_lo + step_lo, step_lo)
print(len(freqs_lo))

for f in freqs_lo:
    vsg.frequency(f)
    print(f)
    time.sleep(1)



### Spectrum Analyzer

test

In [ ]:
f_lo, f_if, n_points = 5.096600e9, 2e6, 20001
vsg.frequency(f_lo)
sa.mode("SA")
sa.measurement("SAN") 
sa.sa.center(f_lo)
sa.sa.span(2.5*f_if)
sa.cont_meas(True)      # single sweep mode
#sa.auto_sweep(True)      # let driver trigger sweep"
sa.sa.npts(n_points)

trace = sa.sa.trace()
freq  = sa.sa.freq_axis()




Text(0, 0.5, 'Power (dBm)')

register measurement

In [35]:
f_lo, f_if, n_points, a_if = 5.096600e9, 100e3, 20001, 0.1
vsg.frequency(f_lo)
sa.sa.center(f_lo)
sa.sa.span(2.5*f_if)
sa.sa.npts(n_points)

In [37]:
meas = Measurement()
meas.register_parameter(sa.sa.freq_axis)
meas.register_parameter(sa.sa.trace, setpoints=(sa.sa.freq_axis,))

with meas.run() as datasaver:
    trace = sa.sa.trace()
    freq  = sa.sa.freq_axis()
    vsg.frequency(f_lo)
    

    datasaver.add_result(
        (sa.sa.freq_axis, freq),
        (sa.sa.trace, trace),
        )

id = datasaver.run_id
plt.figure()
plt.plot(freq, trace)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Power (dBm)")
plt.title(f"LO frequency: {f_lo/1e9:.4f} GHz, IF : {f_if/1e6:.2f} MHz, IF_AMP: {a_if} V, id: {id}")
plt.grid()
plt.show()


Starting experimental run with id: 15. 


In [32]:
plot_by_id(datasaver.run_id)

([<Axes: title={'center': 'Run #11, Experiment mx_cal (out)'}, xlabel='Frequency (GHz)', ylabel='Trace (dB)'>],
 [None])

In [ ]:
sa = rm.open_resource("GPIB0::18::INSTR")
print(sa.query("*IDN?"))

Keysight Technologies,N9032B,MY62500121,A.34.06



In [11]:


power_vsg = 16 
f_lo = 5.096600e9


In [21]:
meas = Measurement()
meas.register_parameter(sa.sa.freq_axis)
meas.register_parameter(sa.sa.trace, setpoints=(sa.sa.freq_axis,))

meas.register_custom_parameter('LO_amp', label='LO_amp', unit='dBm', paramtype='numeric')
meas.register_custom_parameter('LO_freq', label='LO_freq', unit='Hz', paramtype='numeric')
meas.register_custom_parameter('I_freq', label='I_freq', unit='Hz', paramtype='numeric')
meas.register_custom_parameter('Q_freq', label='Q_freq', unit='Hz', paramtype='numeric')
meas.register_custom_parameter('I_amp', label='I_amp', unit='V', paramtype='numeric')
meas.register_custom_parameter('Q_amp', label='Q_amp', unit='V', paramtype='numeric')

f_if, amplitude1, amplitude2 = 200e3, 0.1, 0.1
vsg.power(power_vsg)
vsg.frequency(f_lo)
    
with meas.run() as datasaver:
    
    trace = sa.sa.trace()
    freq  = sa.sa.freq_axis()
    vsg.frequency(f_lo)
    

    datasaver.add_result(
        (sa.sa.freq_axis, freq),
        (sa.sa.trace, trace),
        ('LO_amp', power_vsg),
        ('LO_freq', f_lo),
        ('I_freq', f_if),
        ('Q_freq', f_if),
        ('I_amp', amplitude1),
        ('Q_amp', amplitude2),
        )


plot_by_id(datasaver.run_id)


Starting experimental run with id: 6. 


([<Axes: title={'center': 'Run #6, Experiment mx_cal (out)'}, xlabel='Frequency (kHz)', ylabel='Trace (dB)'>],
 [None])

### Mixer Calibration

In [18]:
sa = rm.open_resource("GPIB0::18::INSTR")
print(sa.query("*IDN?"))

Keysight Technologies,N9032B,MY62500121,A.34.06



In [69]:
span_sa = 3e6
bw_sa = 1e3
ref_level = -30
n_points = 1001

f_lo = 6e9#5.096600e9
vsg.frequency(f_lo)


In [73]:
sa.write(f":FREQ:CENT {f_lo}") 
sa.write(f":FREQ:SPAN {span_sa}") 
sa.write(f":BAND {bw_sa}")
#sa.write(f":DISP:WIND:TRAC:Y:RLEV -{ref_level}")

sa.write(f":SWE:POINTS {n_points}")

18

In [74]:
## adquisition
sa.write(":INIT:IMM")
sa.query("*OPC?")
trace = sa.query_ascii_values(":TRAC:DATA? TRACE1")
trace = np.array(trace)

f_start = float(sa.query(":FREQ:START?"))
f_stop  = float(sa.query(":FREQ:STOP?"))

freq = np.linspace(f_start, f_stop, len(trace))



In [ ]:
#freqa, tracea = freq,trace

In [78]:
plt.figure()
#plt.plot(freqa, tracea)
plt.plot(freq, trace)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Amplitude (dBm)')
plt.title('Spectrum Analyzer Trace')
plt.grid()



oscilloscope

In [15]:
scope = rm.open_resource("USB0::0x049F::0x505C::CN1811002001495::INSTR")
print(scope.query("*IDN?"))

Hantek, DSO4084C, CN1811002001495, 1.0.1


In [11]:
scope.write("DATA:SOURCE CH1")


17

In [ ]:
sa.sa.center(2e9)

In [ ]:
trace = sa.sa.trace()
freq  = sa.sa.freq_axis()

In [ ]:
sa.cont_meas(True)      # single sweep mode
#sa.auto_sweep(True)      # let driver trigger sweep"

In [ ]:
sa = rm.open_resource("GPIB0::18::INSTR")
print(sa.query("*IDN?"))

Keysight Technologies,N9032B,MY62500121,A.34.06

